In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from pydantic import SecretStr
from typing import TypedDict,NotRequired
from langgraph.graph import StateGraph, START, END

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    api_key = SecretStr(os.environ["GOOGLE_API_KEY"]),
)

In [3]:
class blog_state(TypedDict):
    topic: str
    outline: NotRequired[str]
    blog_content: NotRequired[str]


In [4]:
def create_outline(state: blog_state)->blog_state:

    #fetch title
    title = state.get("topic")

    #call llm gen outline
    prompt = f"Generate outline for the blog on the topic: {title}"
    outline  = llm.invoke(prompt).text

    state["outline"] = outline


    #update state 
    return state


In [5]:
def create_blog(state : blog_state) -> blog_state:

    #fetch outline
    outline = state.get("outline")
    title = state.get("topic")

    #call llm gen blog content
    prompt = f"write a detailed blog on the title - {title} using the outline {outline}"
    blog_content  = llm.invoke(prompt).text

    state["blog_content"] = blog_content

    #update state 
    return state

In [6]:
graph = StateGraph(blog_state)

#nodes 
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

# edges 
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow = graph.compile()

final_output = workflow.invoke({"topic":"who is raman ramanathan"})

final_output


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'who is raman ramanathan',
 'outline': 'Here is a comprehensive blog outline for the topic **"Who is Raman Ramanathan?"** \n\nSince "Raman Ramanathan" could refer to a few different notable figures (depending on whether the search intent is about the corporate leader, an academic, or a public figure), this outline is structured to be adaptable, focusing primarily on the most prominent figures with this name (such as corporate executives, academics, or community leaders). \n\n---\n\n# Blog Outline: Who is Raman Ramanathan?\n\n## I. Introduction\n*   **Hook:** Start with a compelling question or statement about how a name often carries weight in the worlds of business, academia, or innovation.\n*   **Context:** Introduce the name "Raman Ramanathan" and acknowledge that while it may refer to a few different accomplished individuals, it most notably points to [insert primary context, e.g., a prominent business leader / technology executive / academic].\n*   **Thesis Statement:** 

In [7]:
print(final_output['outline'])
# print(final_output['blog_content'])

Here is a comprehensive blog outline for the topic **"Who is Raman Ramanathan?"** 

Since "Raman Ramanathan" could refer to a few different notable figures (depending on whether the search intent is about the corporate leader, an academic, or a public figure), this outline is structured to be adaptable, focusing primarily on the most prominent figures with this name (such as corporate executives, academics, or community leaders). 

---

# Blog Outline: Who is Raman Ramanathan?

## I. Introduction
*   **Hook:** Start with a compelling question or statement about how a name often carries weight in the worlds of business, academia, or innovation.
*   **Context:** Introduce the name "Raman Ramanathan" and acknowledge that while it may refer to a few different accomplished individuals, it most notably points to [insert primary context, e.g., a prominent business leader / technology executive / academic].
*   **Thesis Statement:** This blog dives into who Raman Ramanathan is, their professio